# Hybrid RAG Engine (SciFact + ColBERT)

Thin runner over the `hybrid_rag` package.

## 1) Installing Dependencies

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "hybrid_rag").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

# %pip install -e "{ROOT}"
print("Repo root:", ROOT)

## 2) Configuration and Paths

In [ ]:
from hybrid_rag.config import (
    DATASET_NAME, BASE_DIR, INDEX_DIR, BM25_INDEX_PATH, COLBERT_EMB_PATH,
    DOCSTORE_PATH, BM25_K1, BM25_B, BM25_TOP_K, RERANKER_MODEL,
    RERANKER_BATCH_SIZE, RERANKER_TOP_N, RERANKER_MAX_LENGTH, RAG_MODEL,
    RAG_MAX_CONTEXT_TOKENS, RAG_MAX_NEW_TOKENS, RAG_NUM_EVIDENCE_DOCS,
    DEVICE, EVAL_MAX_QUERIES, EVAL_METRICS,
)

## 3) Inspect Dataset Files

In [ ]:
from huggingface_hub import list_repo_files
for name in sorted(list_repo_files("BeIR/scifact", repo_type="dataset")):
    print(name)
for name in sorted(list_repo_files("BeIR/scifact-qrels", repo_type="dataset")):
    print(name)

## 4-9) Package imports

In [ ]:
from hybrid_rag.data import (
    load_corpus, load_queries, load_train_queries, load_qrels,
    load_dataset, save_docstore, load_docstore,
)
from hybrid_rag.bm25 import BM25Retriever
from hybrid_rag.colbert import ColBERTReranker
from hybrid_rag.finetune import (
    FineTuneColBERTReranker, create_training_triplets, run_fine_tuning_loop,
)
from hybrid_rag.rag import RAGModule
from hybrid_rag.pipeline import build_indexes, search
from hybrid_rag.evaluate import run_full_evaluation

## 10) Run Fine-tuning (optional)

In [ ]:
print("Fine-tuning ColBERT (projection layer only)")

if "docs" not in globals() or "doc_map" not in globals():
    print("Loading corpus...")
    if os.path.exists(DOCSTORE_PATH):
        docs = load_docstore()
    else:
        docs = load_corpus(max_docs=None, verbose=True)
        save_docstore(docs)
    doc_map = {doc["doc_id"]: doc for doc in docs}

if "bm25" not in globals():
    print("Loading BM25 index...")
    bm25 = BM25Retriever()
    if os.path.exists(os.path.join(BM25_INDEX_PATH, "doc_ids.json")):
        bm25.load_index()
    else:
        bm25.build_index(docs)
        bm25.save_index()

if "queries" not in globals() or "qrels" not in globals():
    print("Loading test queries and qrels (for reference)...")
    queries = load_queries()
    qrels = load_qrels(split="test")

print("Loading train queries and qrels...")
train_queries = load_train_queries()
train_qrels = load_qrels(split="train", verbose=True)

training_triplets = create_training_triplets(
    train_queries,
    docs,
    train_qrels,
    bm25_retriever=bm25,
    doc_map=doc_map,
    num_negatives=3,
    bm25_top_k_for_negatives=BM25_TOP_K,
)

if training_triplets:
    EPOCHS = 3
    BATCH_SIZE = 4
    VAL_RATIO = 0.15

    n_train = int(len(training_triplets) * (1 - VAL_RATIO))
    total_steps = max(1, (n_train // BATCH_SIZE)) * EPOCHS

    fine_tuned_reranker = FineTuneColBERTReranker()
    fine_tuned_reranker.load_model()
    fine_tuned_reranker.setup_training(
        learning_rate=1e-4,
        total_steps=total_steps,
        warmup_ratio=0.1,
    )

    run_fine_tuning_loop(
        fine_tuned_reranker,
        training_triplets,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        val_ratio=VAL_RATIO,
        patience=2,
    )

    print("To save: fine_tuned_reranker.model.save_pretrained('./fine_tuned_colbert_model')")
else:
    print("No training triplets created. Check that train_qrels is non-empty.")

## 14) Execute Pipeline

In [ ]:
import os
import time
import gc
import torch

print("Phase 1: BM25 index")

bm25_exists = os.path.exists(os.path.join(BM25_INDEX_PATH, "doc_ids.json"))
store_exists = os.path.exists(DOCSTORE_PATH)

if bm25_exists and store_exists:
    print("Index and docstore exist, loading...")
    start_time = time.time()
    docs = load_docstore()
    doc_map = {doc["doc_id"]: doc for doc in docs}
    bm25 = BM25Retriever()
    bm25.load_index()
    print(f"{len(docs):,} docs loaded ({time.time() - start_time:.1f}s)")
    queries = load_queries()
    qrels = load_qrels()

elif bm25_exists and not store_exists:
    print("BM25 index found but docstore.pkl is missing.")
    print("Rebuilding docstore from raw data (BM25 index will be reused)...")
    start_time = time.time()
    docs, queries, qrels = load_dataset(max_docs=None, qrels_split="test", verbose=True)
    save_docstore(docs)
    doc_map = {doc["doc_id"]: doc for doc in docs}
    bm25 = BM25Retriever()
    bm25.load_index()
    print(f"{len(docs):,} docs ready, docstore saved ({time.time() - start_time:.1f}s)")

else:
    print("No index found, building from scratch...")
    start_time = time.time()
    docs, queries, qrels = load_dataset(max_docs=None, qrels_split="test", verbose=True)
    save_docstore(docs)
    doc_map = {doc["doc_id"]: doc for doc in docs}
    bm25 = BM25Retriever()
    bm25.build_index(docs)
    bm25.save_index()
    print(f"{len(docs):,} docs indexed ({time.time() - start_time:.1f}s)")

print("BM25 speed test:")
for query in queries[:5]:
    start_time = time.time()
    results = bm25.retrieve(query["text"], top_k=BM25_TOP_K)
    elapsed = time.time() - start_time
    top_score = results[0][1] if results else 0
    print(f"[{elapsed:.4f}s] {query['text'][:60]} -> {len(results)} results (top: {top_score:.2f})")

print("Phase 2: ColBERT re-ranker")

if "fine_tuned_reranker" in locals() and fine_tuned_reranker.model is not None:
    print("Using previously fine-tuned ColBERTReranker model.")
    reranker = fine_tuned_reranker
else:
    print("Loading a fresh ColBERTReranker model (no fine-tuning applied).")
    reranker = ColBERTReranker()
    reranker.load_model()

print("Phase 2b: ColBERT document embeddings")

embeddings_exist = os.path.exists(COLBERT_EMB_PATH)
force_reencode = (
    "fine_tuned_reranker" in locals()
    and fine_tuned_reranker.model is not None
    and getattr(fine_tuned_reranker, "best_epoch", 0) > 1
)
if "fine_tuned_reranker" in locals() and fine_tuned_reranker.model is not None:
    if not force_reencode:
        print(
            "Fine-tuned model exists but best_epoch <= 1. "
            "Skipping re-encode and using base model embeddings instead."
        )

if embeddings_exist and not force_reencode:
    print(f"Found saved embeddings at {COLBERT_EMB_PATH}")
    print("Loading pre-computed ColBERT embeddings...")
    start_time = time.time()
    reranker.load_embeddings()
    size_mb = os.path.getsize(COLBERT_EMB_PATH) / (1024 * 1024)
    print(
        f"Loaded {len(reranker.doc_embeddings):,} doc embeddings "
        f"({size_mb:.1f} MB, {time.time() - start_time:.1f}s)"
    )
elif force_reencode and embeddings_exist:
    print("Fine-tuned model detected and embeddings exist. Re-encoding with the fine-tuned model...")
    os.remove(COLBERT_EMB_PATH)
    reranker.doc_embeddings = None
else:
    print("No saved embeddings found, or re-encoding was forced. Encoding corpus with ColBERT.")
    print("Offloading RAG model from GPU to free VRAM for encoding...")

    if "rag" not in dir() or rag.model is None:
        rag = RAGModule()

    reranker.model.to("cpu")
    if reranker.linear is not None:
        reranker.linear.to("cpu")
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        free_gb = torch.cuda.mem_get_info()[0] / 1024**3
        print(f"GPU free after offload: {free_gb:.1f} GB")

    reranker.model.to(DEVICE)
    if reranker.linear is not None:
        reranker.linear.to(DEVICE)

    os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
    encode_batch_size = 8
    print(f"Encoding {len(docs):,} documents with ColBERT (runs once, then saved)...")
    start_time = time.time()
    reranker.encode_corpus(docs, batch_size=encode_batch_size)
    reranker.save_embeddings()
    size_mb = os.path.getsize(COLBERT_EMB_PATH) / (1024 * 1024)
    elapsed = time.time() - start_time
    print(f"Embeddings saved to: {COLBERT_EMB_PATH}")
    print(f"Size: {size_mb:.1f} MB Time: {elapsed:.1f}s")
    print("Download this file to reuse embeddings in future sessions.")

    reranker.model.to("cpu")
    if reranker.linear is not None:
        reranker.linear.to("cpu")
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

if "rag" not in dir() or rag.model is None:
    rag = RAGModule()

print("Moving ColBERT re-ranker back to DEVICE for evaluation...")
reranker.model.to(DEVICE)
if reranker.linear is not None:
    reranker.linear.to(DEVICE)
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"GPU free after ColBERT re-activation: {free_gb:.1f} GB")

print("Phase 3: Full evaluation with relevance judgments")

report = run_full_evaluation(
    bm25=bm25,
    reranker=reranker,
    docs=docs,
    doc_map=doc_map,
    queries=queries,
    qrels=qrels,
)
print("Done!")

## Interactive Query

In [ ]:
print("Enter your query below to get a RAG-generated answer.")

if "reranker" in globals() and reranker.model:
    reranker.model.to(DEVICE)
    if reranker.linear:
        reranker.linear.to(DEVICE)

if "rag" in globals() and rag.model is None:
    rag.load_model()

query = input("Your Query> ").strip()

if query:
    print("Processing your query...")
    search(query, bm25, reranker, rag, doc_map)
else:
    print("No query entered. Exiting interactive query mode.")

In [ ]:
print("Sample retrieval and ranking explanation")

sample_query = "role of vitamin d in bone health"
search_results = search(sample_query, bm25, reranker, rag, doc_map)